# ECH Config Occurence count for Distinct ECH Configs under Test_date, with related publicname

In [ ]:
import os
import pickle
from datetime import datetime
from multiprocessing import Pool

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

pd.set_option("display.max_rows", None)  # Show all rows
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.width", 1000)  # Set a larger width for better view if necessary
pd.set_option("display.max_colwidth", None)  # Show full content of each column

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location

testDates = [
    "2024-08-08",
    "2024-08-19",
    "2024-08-21",
    "2024-08-29",
    "2024-09-05",
    "2024-09-06",
    "2024-09-08",
    "2024-09-11",
    "2024-09-15",
    "2024-10-13",
    "2024-10-20",
    "2024-10-29",
    "2024-10-31",
    "2024-11-07",
    "2024-11-12",
    "2024-11-15",
    "2024-11-17",
    "2024-11-18",
    "2024-11-19",
    "2024-11-27",
    "2024-11-28",
    "2024-11-29",
    "2024-11-30",
    "2024-12-01",
    "2024-12-02",
    "2024-12-03",
    "2024-12-04",
    "2024-12-12",
    "2024-12-13",
    "2024-12-14",
    "2024-12-15",
    "2024-12-16",
    "2024-12-17",
    "2024-12-18",
    "2024-12-19",
    "2024-12-20",
    "2024-12-21",
    "2024-12-22",
    "2024-12-23",
    "2024-12-24",
    "2024-12-25",
    "2024-12-26",
    "2024-12-29",
    "2024-12-30",
    "2025-01-01",
    "2025-01-02",
    "2025-01-03",
    "2025-01-04",
    "2025-01-05",
    "2025-01-06",
    "2025-01-07",
    "2025-01-08",
    "2025-01-09",
    "2025-01-10",
    "2025-01-11",
    "2025-01-12",
    "2025-01-13",
    "2025-01-14",
    "2025-01-15",
    "2025-01-16",
    "2025-01-17",
    "2025-01-18",
    "2025-01-19",
    "2025-01-20",
    "2025-01-21",
    "2025-01-22",
    "2025-01-23",
    "2025-01-24",
    "2025-01-25",
    "2025-01-26",
    "2025-01-27",
    "2025-01-28",
    "2025-01-29",
    "2025-01-30",
    "2025-01-31",
    "2025-02-01",
    "2025-02-02",
    "2025-02-03",
    "2025-02-04",
    "2025-02-05",
    "2025-02-06",
    "2025-02-07",
    "2025-02-08",
    "2025-02-09",
    "2025-02-10",
    "2025-02-11",
    "2025-02-12",
    "2025-02-14",
    "2025-02-15",
    "2025-02-16",
    "2025-02-17",
    "2025-02-18",
    "2025-02-19",
]

query = """
SELECT
    hr.ech_config,
    COUNT(*) AS ech_config_occurrence_count,
    ec.ech_public_name,
    hr.test_date
FROM
    public."HTTPSRecords" hr
JOIN
    public."HTTPSRecordsECHConfigs" hrec ON hr.id = hrec.https_record_id
JOIN
    public."ECHConfigs" ec ON hrec.ech_config_id = ec.id
WHERE
    hr.test_date = %s
    AND hr.ech_config != ''
GROUP BY
    hr.ech_config, ec.ech_public_name, hr.test_date
ORDER BY
    ech_config_occurrence_count DESC, hr.ech_config;
"""

# Create engine at the module level
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}",
    connect_args={"connect_timeout": 3600},
    poolclass=QueuePool,
    pool_size=5,
    max_overflow=10,
)


def fetch_data(query, date):
    """Fetches data for the given query and date."""
    print(f"Starting query for date: {date}")
    with engine.connect() as conn:
        params = (date,)
        df = pd.read_sql_query(query, conn, params=params)
    print(f"Finished query for date: {date}")
    return date, df


def fetch_and_save_data(query, filename_suffix, testDates, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling and multiprocessing,
    saves each test date's data to separate pickle files, prints all data as a single dataframe,
    and outputs all filenames for easy copying.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        testDates (list): A list of dates to execute the query for.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        # Create a list to store the tasks for the pool
        tasks = [(query, date) for date in testDates]

        # Use a multiprocessing pool to execute the tasks in parallel
        with Pool() as pool:
            results = pool.starmap(fetch_data, tasks)

        all_dfs = []  # List to store all dataframes
        filenames = []  # List to store all filenames

        # Save each date's data to a separate pickle file
        for date, df in results:
            all_dfs.append(df)  # Add the dataframe to the list

            os.makedirs("./pickle", exist_ok=True)
            filename = f"./pickle/{datetime.now().strftime('%Y-%m-%d_%HH-%MM-%SS')}_{filename_suffix}_{date}.pickle"
            filenames.append(filename)  # Add the filename to the list

            with open(filename, "wb") as f:
                pickle.dump(df, f)

        # Concatenate all dataframes and print
        all_data_df = pd.concat(all_dfs, ignore_index=True)
        print("\nAll data:\n", all_data_df)

        # Print all filenames for easy copying
        print("\nFilenames:\n", "\n".join(filenames))

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    filename_suffix = "ech_config_occurrence_count_by_test_date_parallelized"
    fetch_and_save_data(query, filename_suffix, testDates)

Starting query for date: 2024-08-21Starting query for date: 2024-09-05Starting query for date: 2024-09-06Starting query for date: 2024-09-11Starting query for date: 2024-08-08
Starting query for date: 2024-09-15Starting query for date: 2024-08-29Starting query for date: 2024-10-20
Starting query for date: 2024-11-07Starting query for date: 2024-09-08Starting query for date: 2024-11-12Starting query for date: 2024-11-15Starting query for date: 2024-11-18
Starting query for date: 2024-11-19Starting query for date: 2024-10-31Starting query for date: 2024-10-13Starting query for date: 2024-10-29
Starting query for date: 2024-08-19

Starting query for date: 2024-11-17












Finished query for date: 2024-09-05
Finished query for date: 2024-08-08
Finished query for date: 2024-08-29
Finished query for date: 2024-08-19
Finished query for date: 2024-08-21
Finished query for date: 2024-11-12
Finished query for date: 2024-09-06
Finished query for date: 2024-09-08
Finished query for date: 202